# Практика: линейная и логистическая регрессия

Этот notebook содержит **только задания и подсказки**. Полные решения с пояснениями находятся в файле `03_Решения_линейная_и_логистическая_регрессия.ipynb`.

## Правила работы

- Выполняйте задания последовательно.
- Не меняйте `random_state` и параметры генерации данных: тогда результаты будут воспроизводимыми.
- Тестовую выборку не используйте для подбора гиперпараметров или порога в реальных проектах. Здесь часть действий на test допустима только для учебной демонстрации.
- После числового ответа обязательно напишите короткую интерпретацию.
- Если результат отличается от ожиданий, сначала проверьте разбиение, порядок столбцов и то, на какой выборке обучена каждая операция.

## Критерии хорошего решения

1. Код воспроизводится при последовательном запуске.
2. Есть baseline и корректные метрики.
3. Нет утечки данных: preprocessing находится внутри `Pipeline` там, где это нужно.
4. Выводы отвечают на вопрос задачи, а не просто повторяют числа.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_regression, make_classification, load_iris
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, log_loss, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')


# Задание 1. Линейная регрессия и анализ ошибок

Представьте, что `y` — числовой показатель, который нужно прогнозировать по пяти признакам.

**Что нужно сделать:**

1. Обучите `DummyRegressor(strategy='mean')` и `LinearRegression` на `Xr_train`.
2. Получите прогнозы обеих моделей для `Xr_test`.
3. Выведите таблицу с метриками **MAE, RMSE и R²** для baseline и линейной регрессии.
4. Для линейной регрессии найдите три признака с наибольшими абсолютными значениями коэффициентов.
5. Постройте график остатков `y_test - y_pred` против прогнозов. Добавьте горизонтальную линию нуля.
6. Напишите 3–5 предложений:
   - насколько линейная регрессия лучше baseline;
   - что означает знак одного из коэффициентов;
   - есть ли на графике очевидная нелинейная структура или крупные выбросы.

**Подсказка:** RMSE можно получить как `np.sqrt(mean_squared_error(...))`. Для сортировки коэффициентов по модулю удобен `Series.sort_values(key=np.abs, ascending=False)`.


In [2]:
# Данные для заданий 1 и 2 — не изменяйте параметры генерации
X_reg, y_reg = make_regression(
    n_samples=500,
    n_features=5,
    n_informative=3,
    noise=20,
    random_state=7
)
reg_features = [f'x{i}' for i in range(1, 6)]
X_reg = pd.DataFrame(X_reg, columns=reg_features)

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=RANDOM_STATE
)

print(Xr_train.shape, Xr_test.shape)
display(Xr_train.head())


(375, 5) (125, 5)


,x1,x2,x3,x4,x5
227,1.4976,0.1086,-0.2826,1.5033,1.4382
417,0.2144,-1.4871,-0.5661,0.8513,0.0758
203,-1.7476,-0.4484,0.0403,2.5280,-0.0599
126,-0.6151,1.0210,0.7647,-1.4272,-0.4959
329,-0.1250,-0.8210,0.6161,-0.7740,3.5001


In [3]:
# ВАШ КОД: baseline, LinearRegression, метрики и коэффициенты


In [4]:
# ВАШ КОД: график остатков


**ВАШ ВЫВОД:**

_Напишите здесь интерпретацию метрик, коэффициентов и графика остатков._


# Задание 2. Подбор регуляризации Ridge

Используйте те же `Xr_train`, `Xr_test`, `yr_train`, `yr_test`.

**Что нужно сделать:**

1. Создайте `Pipeline` из `StandardScaler` и `Ridge`.
2. Подберите `alpha` среди `np.logspace(-3, 3, 13)` с помощью `GridSearchCV`.
3. Используйте пятикратную кросс-валидацию `KFold(n_splits=5, shuffle=True, random_state=42)` и scoring `'neg_root_mean_squared_error'`.
4. Выведите:
   - лучшее `alpha`;
   - средний RMSE по кросс-валидации для лучшей модели;
   - MAE, RMSE и R² на test.
5. Сравните результат с обычной линейной регрессией из задания 1.
6. Объясните, почему `StandardScaler` должен находиться внутри pipeline и почему нельзя выбирать `alpha` по test.

**Подсказка:** в `param_grid` имя параметра состоит из имени шага и параметра, например `'model__alpha'`. Scikit-learn максимизирует scoring, поэтому CV RMSE равен `-best_score_`.


In [5]:
# ВАШ КОД: Pipeline, GridSearchCV, лучшие параметры и метрики


**ВАШ ВЫВОД:**

_Сравните Ridge и LinearRegression. Объясните роль кросс-валидации и защиту от утечки данных._


# Задание 3. Логистическая регрессия при дисбалансе классов

Класс `1` — меньшинство. Допустим, это объекты, которые особенно важно обнаружить.

**Что нужно сделать:**

1. Обучите baseline `DummyClassifier(strategy='most_frequent')`.
2. Обучите pipeline: `StandardScaler` + `LogisticRegression(C=1.0, max_iter=2000, random_state=42)`.
3. Для baseline и логистической регрессии вычислите:
   - accuracy;
   - balanced accuracy;
   - precision, recall и F1 для класса 1;
   - ROC AUC;
   - log loss.
4. Выведите матрицу ошибок и `classification_report` для логистической регрессии.
5. Ответьте:
   - почему одной accuracy недостаточно;
   - какие ошибки означают FP и FN в этой задаче;
   - насколько модель превосходит baseline по balanced accuracy и recall.

**Подсказка:** для ROC AUC и log loss нужны вероятности `predict_proba(Xc_test)[:, 1]`, а не дискретные классы. Используйте `zero_division=0` там, где это поддерживается.


In [6]:
# Данные для заданий 3–5 — не изменяйте параметры генерации
X_clf, y_clf = make_classification(
    n_samples=800,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    weights=[0.72, 0.28],
    class_sep=1.0,
    flip_y=0.02,
    random_state=21
)
clf_features = [f'x{i}' for i in range(1, 7)]
X_clf = pd.DataFrame(X_clf, columns=clf_features)

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_clf, y_clf,
    test_size=0.25,
    stratify=y_clf,
    random_state=RANDOM_STATE
)

print('Доля класса 1 в train:', round(yc_train.mean(), 3))
print('Доля класса 1 в test: ', round(yc_test.mean(), 3))


Доля класса 1 в train: 0.285
Доля класса 1 в test:  0.285


In [7]:
# ВАШ КОД: baseline, LogisticRegression и таблица метрик


In [8]:
# ВАШ КОД: матрица ошибок и classification_report


**ВАШ ВЫВОД:**

_Объясните ограничения accuracy и смысл FP/FN. Сравните модель с baseline._


# Задание 4. Выбор порога классификации

Используйте обученную логистическую регрессию и её вероятности для `Xc_test`.

**Что нужно сделать:**

1. Для порогов `[0.15, 0.25, 0.35, 0.50, 0.65, 0.80]` постройте таблицу с accuracy, precision, recall и F1 для класса 1.
2. Среди порогов, где **recall не ниже 0.85**, выберите порог с максимальным precision. Если precision одинаков, выберите больший порог.
3. Постройте матрицу ошибок для выбранного порога.
4. Сравните её с матрицей при пороге 0.50: как изменились FP и FN?
5. Сформулируйте, когда такой критерий выбора порога может быть полезен.

**Важно:** в реальном проекте порог выбирается на validation, а не на test. Здесь test используется, чтобы сосредоточиться на механике расчёта.


In [9]:
# ВАШ КОД: таблица порогов и выбор порога по критерию


In [10]:
# ВАШ КОД: матрица ошибок для выбранного порога


**ВАШ ВЫВОД:**

_Опишите компромисс между precision и recall, изменение FP/FN и возможную область применения критерия._


# Задание 5. Коэффициенты и отношение шансов

Используйте pipeline логистической регрессии из задания 3.

**Что нужно сделать:**

1. Извлеките коэффициенты модели.
2. Постройте таблицу: признак, коэффициент, `exp(coef)`, абсолютное значение коэффициента.
3. Отсортируйте признаки по абсолютному значению коэффициента.
4. Выберите один положительный и один отрицательный коэффициент. Интерпретируйте их с учётом того, что признаки стандартизированы.
5. Объясните, почему `exp(coef)` не следует называть изменением вероятности в процентных пунктах и почему коэффициенты не доказывают причинность.

**Подсказка:** шаг модели можно получить через `named_steps['model']`, если вы дали ему имя `model`.


In [11]:
# ВАШ КОД: таблица коэффициентов и odds ratio


**ВАШ ВЫВОД:**

_Интерпретируйте два коэффициента и укажите ограничения такой интерпретации._


# Задание 6 (дополнительное). Мультиклассовая классификация Iris

Теперь целевая переменная содержит три класса iris.

**Что нужно сделать:**

1. Разделите данные на train/test в соотношении 75/25 со `stratify=y` и `random_state=42`.
2. Обучите pipeline `StandardScaler` + `LogisticRegression(C=1.0, max_iter=2000, random_state=42)`.
3. Вычислите accuracy и **macro F1**.
4. Выведите `classification_report` и матрицу ошибок.
5. Объясните:
   - чем эта задача отличается от бинарной классификации;
   - что усредняет macro F1;
   - какие классы модель путает, если ошибки есть.

**Подсказка:** `load_iris(return_X_y=True, as_frame=True)` возвращает признаки и целевую переменную.


In [12]:
# ВАШ КОД: мультиклассовая логистическая регрессия


**ВАШ ВЫВОД:**

_Опишите отличие от бинарной задачи, смысл macro F1 и ошибки модели._


# Финальная самопроверка

Перед сравнением с решением проверьте:

- [ ] Для регрессии есть baseline, MAE, RMSE и R².
- [ ] Коэффициенты интерпретируются с учётом масштаба признаков.
- [ ] Ridge настроен только через кросс-валидацию на train.
- [ ] Для классификации использованы вероятности там, где это требуется.
- [ ] Метрики класса 1 не перепутаны с усреднёнными метриками.
- [ ] Порог превращает вероятность в класс через `proba >= threshold`.
- [ ] Каждый числовой результат сопровождается выводом.
- [ ] Все ячейки запускаются сверху вниз без ошибок.

После самостоятельной попытки откройте `03_Решения_линейная_и_логистическая_регрессия.ipynb` и сравните не только ответы, но и ход рассуждений.
